# Model evaluation

In this notebook:
* We plot the results of the inference produced in 09_3_predict.
* We evaluate the model performance visually and by comparing the predictions to the ground truth.
* We compute model performance metrics.

Used labels:
* 0 - Boats

Ideas and codesnippets from: 
* https://mayrajeo.github.io/ship-detection/
* https://docs.ultralytics.com/guides/sahi-tiled-inference/


In [ ]:
# General libraries
import sys, os
from pathlib import Path

# Torch
import torch

# Pandas
import geopandas as gpd
import pandas as pd

# Shapely
from shapely.geometry import box, Polygon, MultiPolygon
import shapely

# Matplotlib for plotting
import matplotlib.pyplot as plt
import matplotlib.image as img

# Rasterio
import rasterio
import rasterio.plot as rioplot
from rasterio.plot import show
from rasterio.windows import from_bounds

# Ultralytics
from ultralytics import YOLO

# Folium
import folium

# json
import json

In [ ]:
# Get user
user = os.environ.get("USER") 

# Set path to the exercise directory
exercise_folder = os.path.join(os.sep, 'scratch', 'project_462001167', 'students', os.environ.get('USER'), 'GeoML', '09_object_detection', 'own_model_training') 

# Set path to the ground truth file
test_file = os.path.join(exercise_folder, 'annotations','34VEN.gpkg')

# Set layername for the ground truth file
test_layer = '20210714'

# Set path to the test image, Sentinel-2 raster
sentinel_raster = os.path.join(exercise_folder, 'sentinel2','T34VEN_20210714T100029_TCI.tif')

# Set path to the annotated image produced in the previous step that we want to visualize
prediction_file = os.path.join(exercise_folder, 'predictions', 'T34VEN_20210714T100029_TCI.geojson')
annotation_image = os.path.join(exercise_folder, 'predictions', 'T34VEN_20210714T100029_TCI_predictions.png')

# Set bbox for the test and plot areas
test_area = (500666.1474717269884422, 6700020.0, 575548.6236228030174971, 6799586.8574843602254987)
plot_area = (523000, 6703700, 529000, 6707806)

# Set path to metrics JSON file
json_path = os.path.join(exercise_folder, "metrics.json")

# Visualize predictions

### Full test image plot

It is relatively difficult to plot raster images and vector data with Jupyter, so consider using QGIS or ArcGIS for viewing the data interactively.

This image was prepared with `09_3_predict.py` script.

The predicted boats are the small blue dots.

This is image in .png format (not georeferenced), so you can open it also with JupyterLab to get better zoom.

In [ ]:
im = img.imread(annotation_image)

plt.figure(figsize=(20, 20))  # width, height in inches
plt.imshow(im)
plt.axis('off')
plt.show()

### Zoomable map

* Ground truth - orange
* Predicted boats - yellow

The boats are small, so difficult to see. Zoom in along the coast.

In [ ]:
prediction = gpd.read_file(prediction_file, bbox=test_area) # The test datasets does not include data for full Sentinel image, so select the part it covers.
test = gpd.read_file(test_file, layer=test_layer)

In [ ]:
m = test.explore(
    name="test",  # name of the layer in the map
    color="red",  # use red color on all points    
)

prediction.explore(
    m=m,  # pass the map object
    color="orange",  # use red color on all points
    name="prediction",  # name of the layer in the map
)

### Plot of a smaller bbox

Zoom in to a smaller area, to better see the results.

Test data (red) and predictions (yellow) on top the used Sentinel-2 image.

In [ ]:
test_plot = gpd.read_file(test_file, layer=test_layer, bbox=plot_area)
prediction_plot = gpd.read_file(prediction_file, bbox=plot_area)

with rasterio.open(sentinel_raster) as src:
    window = from_bounds(plot_area[0], plot_area[1], plot_area[2], plot_area[3], src.transform)
    raster = src.read(window=window)
    transform = src.window_transform(window)

    fig, ax = plt.subplots(figsize=(15, 15))
    show(raster, transform=transform, ax=ax)
    test_plot.plot(ax=ax, facecolor='none', edgecolor='red')
    prediction_plot.plot(ax=ax, facecolor='none', edgecolor='yellow')

Note, that we are mainly missing predictions, so it might be better to use even lower prediction threshold.

## Count true positives, true negatives and false positives with intersection

Boats are very small objects in Sentinel-images, so exact bbox is less important than the object was found at all. So here we count true positives, true negatives and false positives with simple intersection. Note, that in ML context, this is not likely very usual.

Calculate for predicted boats:
* Correcty predicted boats (true positives) - predicted boat polygon overlaps with a polygon in test data
* Not predicted boats (true negatives) - no prediction polygon in the area of a boat in test data
* Predicted boats, that are not boats (false positives) - predicted boat polygon does not overlap with any polygon in test data

In [ ]:


joined = gpd.sjoin(test, prediction, how='left', predicate='intersects')

# Keep only polygons with no match (i.e., no overlap)
correct = joined[joined.index_right.notna()].drop(columns='index_right')
missing = joined[joined.index_right.isna()].drop(columns='index_right')

joined2 = gpd.sjoin(prediction, test, how='left', predicate='intersects')
false = joined2[joined2.index_right.isna()].drop(columns='index_right')

print('Boats in test data: ' + str(len(test)))
print('Predicted boats: ' + str(len(prediction)))
print('Correctly predicted boats  (true positives): ' + str(len(correct)))
print('Not predicted boats  (true negatives): ' + str(len(missing)))
print('Predicted boats, that are not boats (false positives): ' + str(len(false)))

Plot the different categories:
* Correcty predicted boats - blue
* Not predicted boats - orange
* Predicted boats, that are not boats - red

In [ ]:
m = correct.explore(
    name="correct",  # name of the layer in the map
    color="blue",  # use red color on all points 
)

missing.explore(
    m=m,  # pass the map object
    color="orange",  # use red color on all points
    name="missing",  # name of the layer in the map
)

false.explore(
    m=m,  # pass the map object
    color="red",  # use red color on all points
    name="false",  # name of the layer in the map
    #weight=6,

)

folium.LayerControl().add_to(m)  # use folium to add layer control
m

# Performance metrics

Show the model performance metrics calculated with the test data with `09_3_predict.py` script. 

In [ ]:
with open(json_path, "r") as file:
    metrics = json.load(file)

metrics